In [ ]:
%load_ext cudf.pandas

In [ ]:
import numpy as np
import os
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    import os
    os.environ["MODIN_ENGINE"] = "ray"
    import ray
    ray.init(num_cpus=int(os.environ['MODIN_CPUS']), runtime_env={'env_vars': {'__MODIN_AUTOIMPORT_PANDAS__': '1'}})
    import modin.pandas as pd
else:
    import pandas as pd
from glob import glob
import matplotlib.pyplot as plt
import matplotlib.style as style
style.use('fivethirtyeight')
from matplotlib.ticker import FuncFormatter
from nltk.corpus import stopwords
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')
from sklearn.feature_extraction.text import CountVectorizer
import os

In [ ]:
%%time
### cell 0 ###

factor = 50000
# Read only the first 1000 rows and cast columns to integer on the GPU in one step
train = pd.read_csv(
    '/home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/input/feedback-prize-2021/train.csv',
    nrows=factor,
    dtype={
        'discourse_id': 'int64',
        'discourse_start': 'int64',
        'discourse_end': 'int64'
    }
)

# Sample submission still fits in memory, read normally
sample_submission = pd.read_csv(
    '/home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/input/feedback-prize-2021/sample_submission.csv'
)

# File‐listing remains on the CPU
train_txt = glob('/home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/input/feedback-prize-2021/train/*.txt')
test_txt  = glob('/home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/input/feedback-prize-2021/test/*.txt')

In [ ]:

import cudf
try:
    pd_train = cudf.DataFrame(index=train.index).to_pandas()
except:
    pd_train = cudf.DataFrame().to_pandas()
                

In [ ]:
%%time
## Transfer_post 0 ##
pd_train["""id"""] = train["""id"""].to_numpy()

In [ ]:
%%time
## Transfer_post 0 ##
pd_train["""discourse_id"""] = train["""discourse_id"""].to_numpy()

In [ ]:
%%time
## Transfer_post 0 ##
pd_train["""discourse_start"""] = train["""discourse_start"""].to_numpy()

In [ ]:
%%time
## Transfer_post 0 ##
pd_train["""discourse_end"""] = train["""discourse_end"""].to_numpy()

In [ ]:
%%time
## Transfer_post 0 ##
pd_train["""discourse_text"""] = train["""discourse_text"""].to_numpy()

In [ ]:
%%time
## Transfer_post 0 ##
pd_train["""discourse_type"""] = train["""discourse_type"""].to_numpy()

In [ ]:
%%time
## Transfer_post 0 ##
pd_train["""discourse_type_num"""] = train["""discourse_type_num"""].to_numpy()

In [ ]:
%%time
## Transfer_post 0 ##
pd_train["""predictionstring"""] = train["""predictionstring"""].to_numpy()

In [ ]:

import cudf
try:
    pd_train = cudf.DataFrame(index=train.index).to_pandas()
except:
    pd_train = cudf.DataFrame().to_pandas()
                

In [ ]:
%%time
## Transfer_pre 1 ##
pd_train["""id"""] = train["""id"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 1 ##
pd_train["""discourse_id"""] = train["""discourse_id"""].to_numpy()

In [ ]:
%%time
### cell 1 ###

sample_discourse_id = train.loc[train.id == "423A1CA112E2", "discourse_id"].iloc[0]

In [ ]:

import cudf
try:
    pd_train = cudf.DataFrame(index=train.index).to_pandas()
except:
    pd_train = cudf.DataFrame().to_pandas()
                

In [ ]:
%%time
## Transfer_pre 2 ##
pd_train["""predictionstring"""] = train["""predictionstring"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 2 ##
pd_train["""discourse_type"""] = train["""discourse_type"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 2 ##
pd_train["""discourse_id"""] = train["""discourse_id"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 2 ##
pd_train["""discourse_text"""] = train["""discourse_text"""].to_numpy()

In [ ]:
%%time
### cell 2 ###

#add columns
train["discourse_len"] = train["discourse_text"].apply(lambda x: len(x.split()))
train["pred_len"] = train["predictionstring"].apply(lambda x: len(x.split()))

cols_to_display = ['discourse_id', 'discourse_text', 'discourse_type','predictionstring', 'discourse_len', 'pred_len']
train[cols_to_display].head()

In [ ]:

import cudf
try:
    pd_train = cudf.DataFrame(index=train.index).to_pandas()
except:
    pd_train = cudf.DataFrame().to_pandas()
                

In [ ]:
%%time
## Transfer_post 2 ##
pd_train["""pred_len"""] = train["""pred_len"""].to_numpy()

In [ ]:
%%time
## Transfer_post 2 ##
pd_train["""discourse_len"""] = train["""discourse_len"""].to_numpy()

In [ ]:

import cudf
try:
    pd_train = cudf.DataFrame(index=train.index).to_pandas()
except:
    pd_train = cudf.DataFrame().to_pandas()
                

In [ ]:
%%time
## Transfer_pre 3 ##
pd_train["""pred_len"""] = train["""pred_len"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 3 ##
pd_train["""discourse_len"""] = train["""discourse_len"""].to_numpy()

In [ ]:
%%time
### cell 3 ###

print(f"The total number of discourses is {len(train)}")
train[train.discourse_len != train.pred_len][cols_to_display]

In [ ]:

import cudf
try:
    pd_train = cudf.DataFrame(index=train.index).to_pandas()
except:
    pd_train = cudf.DataFrame().to_pandas()
                

In [ ]:
%%time
## Transfer_pre 4 ##
pd_train["""discourse_id"""] = train["""discourse_id"""].to_numpy()

In [ ]:
%%time
### cell 4 ###

# Optimized cell 4 for cudf
def print_discourse_info(train, sample_discourse_id):
    # Use GPU‐accelerated boolean indexing instead of query
    mask = train.discourse_id == sample_discourse_id
    subset = train[mask]
    # Retrieve the text once
    text = subset['discourse_text'].iloc[0]
    # Split on CPU (single string) or use cudf string split if desired
    tokens = text.split()
    print(text)
    print(tokens)
    print(len(tokens))

print_discourse_info(train, sample_discourse_id)

In [ ]:

import cudf
try:
    pd_train = cudf.DataFrame(index=train.index).to_pandas()
except:
    pd_train = cudf.DataFrame().to_pandas()
                

In [ ]:
%%time
## Transfer_pre 5 ##
pd_train["""predictionstring"""] = train["""predictionstring"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 5 ##
pd_train["""discourse_id"""] = train["""discourse_id"""].to_numpy()

In [ ]:
%%time
### cell 5 ###

# Fetch the prediction string as a Python scalar (1 GPU lookup + head)
pred_str = train['predictionstring'][train['discourse_id'] == sample_discourse_id] \
             .head(1) \
             .item()

# Split and length on the CPU for a single small string (cheap)
pred_list = pred_str.split(' ')
pred_len  = len(pred_list)

print(pred_str)
print(pred_list)
print(pred_len)

In [ ]:

import cudf
try:
    pd_train = cudf.DataFrame(index=train.index).to_pandas()
except:
    pd_train = cudf.DataFrame().to_pandas()
                

In [ ]:
%%time
## Transfer_pre 6 ##
pd_train["""discourse_type"""] = train["""discourse_type"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 6 ##
pd_train["""discourse_len"""] = train["""discourse_len"""].to_numpy()

In [ ]:
%%time
### cell 6 ###

# Combine aggregations to a single groupby to minimize GPU passes
agg = train.groupby('discourse_type').agg({
    'discourse_len': 'mean',        # mean length per discourse_type
    'discourse_type': 'count'      # count per discourse_type
})
ax1 = agg['discourse_len'].sort_values()
ax2 = agg['discourse_type'].sort_values()

In [ ]:

import cudf
try:
    pd_train = cudf.DataFrame(index=train.index).to_pandas()
except:
    pd_train = cudf.DataFrame().to_pandas()
                

In [ ]:
%%time
## Transfer_pre 7 ##
pd_train["""discourse_type_num"""] = train["""discourse_type_num"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 7 ##
pd_train["""id"""] = train["""id"""].to_numpy()

In [ ]:
%%time
### cell 7 ###

# cell 7 optimized
total_essays = train.id.nunique()
# compute percentage per discourse type in one shot, avoid intermediate DataFrame
perc = (train.discourse_type_num.value_counts(ascending=True) / total_essays).round(3)
perc.name = 'perc'
ax = perc[perc > 0.03]

In [ ]:

import cudf
try:
    pd_train = cudf.DataFrame(index=train.index).to_pandas()
except:
    pd_train = cudf.DataFrame().to_pandas()
                

In [ ]:
%%time
## Transfer_pre 8 ##
pd_train["""discourse_type"""] = train["""discourse_type"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 8 ##
pd_train["""discourse_end"""] = train["""discourse_end"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 8 ##
pd_train["""discourse_start"""] = train["""discourse_start"""].to_numpy()

In [ ]:
%%time
### cell 8 ###

data = (
    train.groupby("discourse_type", as_index=False)[["discourse_end", "discourse_start"]]
    .mean()
    .sort_values(by="discourse_start", ascending=False)
)

In [ ]:

import cudf
try:
    pd_data = cudf.DataFrame(index=data.index).to_pandas()
except:
    pd_data = cudf.DataFrame().to_pandas()
                

In [ ]:
%%time
## Transfer_post 8 ##
pd_data["""discourse_type"""] = data["""discourse_type"""].to_numpy()

In [ ]:
%%time
## Transfer_post 8 ##
pd_data["""discourse_end"""] = data["""discourse_end"""].to_numpy()

In [ ]:
%%time
## Transfer_post 8 ##
pd_data["""discourse_start"""] = data["""discourse_start"""].to_numpy()

In [ ]:

import cudf
try:
    pd_train = cudf.DataFrame(index=train.index).to_pandas()
except:
    pd_train = cudf.DataFrame().to_pandas()
                

In [ ]:
%%time
## Transfer_pre 9 ##
pd_train["""id"""] = train["""id"""].to_numpy()

In [ ]:
%%time
### cell 9 ###

# 1) Compute number of unique essays once
n_ids = train.id.nunique()

# 2) First discourse_type per essay
train_first = (
    train
    .drop_duplicates(subset='id', keep='first')
    .discourse_type
    .value_counts()
)
train_first.index.name = 'discourse_type'
train_first = train_first.reset_index(name='counts_first')
train_first['percent_first'] = round(train_first['counts_first'] / n_ids, 2)

# 3) Last discourse_type per essay
train_last = (
    train
    .drop_duplicates(subset='id', keep='last')
    .discourse_type
    .value_counts()
)
train_last.index.name = 'discourse_type'
train_last = train_last.reset_index(name='counts_last')
train_last['percent_last'] = round(train_last['counts_last'] / n_ids, 2)

# 4) Merge first and last counts
train_first_last = train_first.merge(
    train_last,
    on='discourse_type',
    how='left'
)

train_first_last

In [ ]:

import cudf
try:
    pd_train = cudf.DataFrame(index=train.index).to_pandas()
except:
    pd_train = cudf.DataFrame().to_pandas()
                

In [ ]:
%%time
## Transfer_pre 10 ##
pd_train["""discourse_type"""] = train["""discourse_type"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 10 ##
pd_train["""id"""] = train["""id"""].to_numpy()

In [ ]:
%%time
### cell 10 ###

# Vectorized assignment of discourse_nr using GPU-accelerated groupby cumcount
train['discourse_nr'] = train.groupby('id').cumcount() + 1

# Use boolean indexing instead of .query to stay on GPU
train[train['discourse_type'].isin(['Lead'])]  \
     .groupby('discourse_type_num')['discourse_nr']

In [ ]:

import cudf
try:
    pd_train = cudf.DataFrame(index=train.index).to_pandas()
except:
    pd_train = cudf.DataFrame().to_pandas()
                

In [ ]:
%%time
## Transfer_post 10 ##
pd_train["""discourse_nr"""] = train["""discourse_nr"""].to_numpy()

In [ ]:

import cudf
try:
    pd_train = cudf.DataFrame(index=train.index).to_pandas()
except:
    pd_train = cudf.DataFrame().to_pandas()
                

In [ ]:
%%time
## Transfer_pre 11 ##
pd_train["""id"""] = train["""id"""].to_numpy()

In [ ]:
%%time
### cell 11 ###

# Step 1: read all essays into Python lists
ids = []
texts = []
for path in tqdm(train_txt):
    with open(path, 'r', encoding='utf-8') as f:
        text = f.read()
    ids.append(path.rsplit('/', 1)[-1].replace('.txt', ''))
    texts.append(text)

# Step 2: build a GPU-backed DataFrame via pandas shim
# (the  extension makes pandas.DataFrame GPU-backed)
df_text = pd.DataFrame({'id': ids, 'text': texts})

# Step 3: vectorized string ops + cast to int64 to match original dtypes
df_text['essay_len'] = (
    df_text['text']
      .str.strip()
      .str.len()
      .astype('int64')
)
df_text['essay_words'] = (
    df_text['text']
      .str.split()
      .apply(len)
      .astype('int64')
)

# Step 4: merge back into `train` and assign
tmp = train.merge(
    df_text[['id', 'essay_len', 'essay_words']],
    on='id', how='left'
)
train['essay_len'] = tmp['essay_len']
train['essay_words'] = tmp['essay_words']

In [ ]:

import cudf
try:
    pd_train = cudf.DataFrame(index=train.index).to_pandas()
except:
    pd_train = cudf.DataFrame().to_pandas()
                

In [ ]:
%%time
## Transfer_post 11 ##
pd_train["""essay_len"""] = train["""essay_len"""].to_numpy()

In [ ]:
%%time
## Transfer_post 11 ##
pd_train["""essay_words"""] = train["""essay_words"""].to_numpy()

In [ ]:

import cudf
try:
    pd_train = cudf.DataFrame(index=train.index).to_pandas()
except:
    pd_train = cudf.DataFrame().to_pandas()
                

In [ ]:
%%time
## Transfer_pre 12 ##
pd_train["""discourse_type"""] = train["""discourse_type"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 12 ##
pd_train["""predictionstring"""] = train["""predictionstring"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 12 ##
pd_train["""id"""] = train["""id"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 12 ##
pd_train["""discourse_id"""] = train["""discourse_id"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 12 ##
pd_train["""discourse_text"""] = train["""discourse_text"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 12 ##
pd_train["""discourse_type_num"""] = train["""discourse_type_num"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 12 ##
pd_train["""discourse_end"""] = train["""discourse_end"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 12 ##
pd_train["""discourse_start"""] = train["""discourse_start"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 12 ##
pd_train["""discourse_len"""] = train["""discourse_len"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 12 ##
pd_train["""essay_words"""] = train["""essay_words"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 12 ##
pd_train["""pred_len"""] = train["""pred_len"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 12 ##
pd_train["""essay_len"""] = train["""essay_len"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 12 ##
pd_train["""discourse_nr"""] = train["""discourse_nr"""].to_numpy()

In [ ]:
%%time
### cell 12 ###

# Vectorized computation of gap_length without Python loops
prev_id = train['id'].shift(1)
prev_end = train['discourse_end'].shift(1)
curr_id = train['id']
curr_start = train['discourse_start']

cond_continuous = (curr_id == prev_id) & ((curr_start - prev_end) > 1)
cond_new_essay = (curr_id != prev_id) & (curr_start != 0)

train['gap_length'] = np.where(
    cond_continuous,
    curr_start - prev_end - 2,
    np.where(cond_new_essay, curr_start - 1, np.nan)
)

# Ensure the very first row matches the original hard-coded value
train.loc[train.index[0], 'gap_length'] = curr_start.iloc[0] - 1

# Compute gap at end of each essay and merge back
last_ones = train.drop_duplicates(subset='id', keep='last')
last_ones['gap_end_length'] = np.where(
    last_ones['discourse_end'] < last_ones['essay_len'],
    last_ones['essay_len'] - last_ones['discourse_end'],
    np.nan
)
cols_to_merge = ['id', 'discourse_id', 'gap_end_length']
train = train.merge(last_ones[cols_to_merge], on=['id', 'discourse_id'], how='left')

In [ ]:

import cudf
try:
    pd_train = cudf.DataFrame(index=train.index).to_pandas()
except:
    pd_train = cudf.DataFrame().to_pandas()
                

In [ ]:
%%time
## Transfer_pre 13 ##
pd_train["""discourse_type"""] = train["""discourse_type"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 13 ##
pd_train["""id"""] = train["""id"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 13 ##
pd_train["""gap_end_length"""] = train["""gap_end_length"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 13 ##
pd_train["""gap_length"""] = train["""gap_length"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 13 ##
pd_train["""discourse_end"""] = train["""discourse_end"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 13 ##
pd_train["""discourse_start"""] = train["""discourse_start"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 13 ##
pd_train["""essay_len"""] = train["""essay_len"""].to_numpy()

In [ ]:
%%time
### cell 13 ###

#display an example
cols_to_display = ['id', 'discourse_start', 'discourse_end', 'discourse_type',
                   'essay_len', 'gap_length', 'gap_end_length']
# use GPU‐accelerated boolean indexing instead of .query()
train.loc[train['id'] == 'AFEC37C2D43F', cols_to_display]

In [ ]:

import cudf
try:
    pd_train = cudf.DataFrame(index=train.index).to_pandas()
except:
    pd_train = cudf.DataFrame().to_pandas()
                

In [ ]:
%%time
## Transfer_pre 14 ##
pd_train["""gap_end_length"""] = train["""gap_end_length"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 14 ##
pd_train["""gap_length"""] = train["""gap_length"""].to_numpy()

In [ ]:
%%time
### cell 14 ###

# how many pieces of text are not used as discourses?
n_total = len(train)
n_not_classified = train['gap_length'].notna().sum() + train['gap_end_length'].notna().sum()
print(f"Besides the {n_total} discourse texts, there are {n_not_classified} pieces of text not classified.")

In [ ]:

import cudf
try:
    pd_train = cudf.DataFrame(index=train.index).to_pandas()
except:
    pd_train = cudf.DataFrame().to_pandas()
                

In [ ]:
%%time
## Transfer_pre 15 ##
pd_train["""gap_length"""] = train["""gap_length"""].to_numpy()

In [ ]:
%%time
### cell 15 ###

IREWR_tmp = train.sort_values(by = "gap_length", ascending = False)[cols_to_display].head()
IREWR_plug_2 = IREWR_tmp.iloc[0]["id"]
IREWR_tmp

In [ ]:

import cudf
try:
    pd_train = cudf.DataFrame(index=train.index).to_pandas()
except:
    pd_train = cudf.DataFrame().to_pandas()
                

In [ ]:
%%time
## Transfer_pre 16 ##
pd_train["""gap_end_length"""] = train["""gap_end_length"""].to_numpy()

In [ ]:
%%time
### cell 16 ###

IREWR_tmp2 = train.sort_values(by = "gap_end_length", ascending = False)[cols_to_display].head()
IREWR_plug_1 = IREWR_tmp2.iloc[1]["id"]
IREWR_tmp2

In [ ]:

import cudf
try:
    pd_train = cudf.DataFrame(index=train.index).to_pandas()
except:
    pd_train = cudf.DataFrame().to_pandas()
                

In [ ]:
%%time
## Transfer_pre 17 ##
pd_train["""gap_end_length"""] = train["""gap_end_length"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 17 ##
pd_train["""gap_length"""] = train["""gap_length"""].to_numpy()

In [ ]:
%%time
### cell 17 ###

# concatenate and filter in one pass (nulls are excluded by the <300 mask)
all_gaps = pd.concat(
    [train.gap_length, train.gap_end_length],
    ignore_index=True
)
all_gaps = all_gaps[all_gaps < 300]

In [ ]:

import cudf
try:
    pd_train = cudf.DataFrame(index=train.index).to_pandas()
except:
    pd_train = cudf.DataFrame().to_pandas()
                

In [ ]:
%%time
## Transfer_pre 18 ##
pd_train["""essay_len"""] = train["""essay_len"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 18 ##
pd_train["""id"""] = train["""id"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 18 ##
pd_train["""gap_end_length"""] = train["""gap_end_length"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 18 ##
pd_train["""gap_length"""] = train["""gap_length"""].to_numpy()

In [ ]:
%%time
### cell 18 ###

total_gaps = train.groupby('id').agg({'essay_len': 'first',\
                                               'gap_length': 'sum',\
                                               'gap_end_length': 'sum'})
total_gaps['perc_not_classified'] = round(((total_gaps.gap_length + total_gaps.gap_end_length)/total_gaps.essay_len),2)

total_gaps.sort_values(by = 'perc_not_classified', ascending = False).head()

In [ ]:

import cudf
try:
    pd_train = cudf.DataFrame(index=train.index).to_pandas()
except:
    pd_train = cudf.DataFrame().to_pandas()
                

In [ ]:
%%time
## Transfer_pre 19 ##
pd_train["""discourse_type"""] = train["""discourse_type"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 19 ##
pd_train["""predictionstring"""] = train["""predictionstring"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 19 ##
pd_train["""id"""] = train["""id"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 19 ##
pd_train["""gap_end_length"""] = train["""gap_end_length"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 19 ##
pd_train["""discourse_id"""] = train["""discourse_id"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 19 ##
pd_train["""discourse_text"""] = train["""discourse_text"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 19 ##
pd_train["""gap_length"""] = train["""gap_length"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 19 ##
pd_train["""discourse_end"""] = train["""discourse_end"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 19 ##
pd_train["""discourse_type_num"""] = train["""discourse_type_num"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 19 ##
pd_train["""discourse_start"""] = train["""discourse_start"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 19 ##
pd_train["""discourse_len"""] = train["""discourse_len"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 19 ##
pd_train["""essay_words"""] = train["""essay_words"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 19 ##
pd_train["""pred_len"""] = train["""pred_len"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 19 ##
pd_train["""essay_len"""] = train["""essay_len"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 19 ##
pd_train["""discourse_nr"""] = train["""discourse_nr"""].to_numpy()

In [ ]:
%%time
### cell 19 ###

def add_gap_rows(essay):
    cols_to_keep = ['discourse_start', 'discourse_end', 'discourse_type', 'gap_length', 'gap_end_length']
    df_essay = train.query(f"id == '{essay}'")[cols_to_keep].reset_index(drop = True)
    
    print(df_essay)

    #index new row
    insert_row = len(df_essay)
   
    for i_2 in range(1, len(df_essay)):          
        if df_essay.loc[i_2,"gap_length"] >0:
            if i_2 == 0:
                start = 0 #as there is no i-1 for first row
                end = df_essay.loc[0, 'discourse_start'] -1
                disc_type = "Nothing"
                gap_end = np.nan
                gap = np.nan
                df_essay.loc[insert_row] = [start, end, disc_type, gap, gap_end]
                insert_row += 1
            else:
                start = df_essay.loc[i_2-1, "discourse_end"] + 1
                end = df_essay.loc[i_2, 'discourse_start'] -1
                disc_type = "Nothing"
                gap_end = np.nan
                gap = np.nan
                df_essay.loc[insert_row] = [start, end, disc_type, gap, gap_end]
                insert_row += 1

    df_essay = df_essay.sort_values(by = "discourse_start").reset_index(drop=True)

    #add gap at end
    if df_essay.loc[(len(df_essay)-1),'gap_end_length'] > 0:
        start = df_essay.loc[(len(df_essay)-1), "discourse_end"] + 1
        end = start + df_essay.loc[(len(df_essay)-1), 'gap_end_length']
        disc_type = "Nothing"
        gap_end = np.nan
        gap = np.nan
        df_essay.loc[insert_row] = [start, end, disc_type, gap, gap_end]
        
    return(df_essay)
add_gap_rows(IREWR_plug_1)

In [ ]:

import cudf
try:
    pd_data = cudf.DataFrame(index=data.index).to_pandas()
except:
    pd_data = cudf.DataFrame().to_pandas()
                

In [ ]:
%%time
## Transfer_pre 20 ##
pd_data["""discourse_type"""] = data["""discourse_type"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 20 ##
pd_data["""discourse_end"""] = data["""discourse_end"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 20 ##
pd_data["""discourse_start"""] = data["""discourse_start"""].to_numpy()

In [ ]:

import cudf
try:
    pd_train = cudf.DataFrame(index=train.index).to_pandas()
except:
    pd_train = cudf.DataFrame().to_pandas()
                

In [ ]:
%%time
## Transfer_pre 20 ##
pd_train["""discourse_type"""] = train["""discourse_type"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 20 ##
pd_train["""predictionstring"""] = train["""predictionstring"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 20 ##
pd_train["""id"""] = train["""id"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 20 ##
pd_train["""gap_end_length"""] = train["""gap_end_length"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 20 ##
pd_train["""discourse_id"""] = train["""discourse_id"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 20 ##
pd_train["""discourse_text"""] = train["""discourse_text"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 20 ##
pd_train["""gap_length"""] = train["""gap_length"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 20 ##
pd_train["""discourse_end"""] = train["""discourse_end"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 20 ##
pd_train["""discourse_type_num"""] = train["""discourse_type_num"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 20 ##
pd_train["""discourse_start"""] = train["""discourse_start"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 20 ##
pd_train["""discourse_len"""] = train["""discourse_len"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 20 ##
pd_train["""essay_words"""] = train["""essay_words"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 20 ##
pd_train["""pred_len"""] = train["""pred_len"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 20 ##
pd_train["""essay_len"""] = train["""essay_len"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 20 ##
pd_train["""discourse_nr"""] = train["""discourse_nr"""].to_numpy()

In [ ]:
%%time
### cell 20 ###

def add_gap_rows(essay):
    cols_to_keep = ['discourse_start', 'discourse_end', 'discourse_type', 'gap_length', 'gap_end_length']
    df_essay = train.query(f"id == '{essay}'")[cols_to_keep].reset_index(drop = True)
    
    print(df_essay)

    #index new row
    insert_row = len(df_essay)
   
    for i_2 in range(1, len(df_essay)):          
        if df_essay.loc[i_2,"gap_length"] >0:
            if i_2 == 0:
                start = 0 #as there is no i-1 for first row
                end = df_essay.loc[0, 'discourse_start'] -1
                disc_type = "Nothing"
                gap_end = np.nan
                gap = np.nan
                df_essay.loc[insert_row] = [start, end, disc_type, gap, gap_end]
                insert_row += 1
            else:
                start = df_essay.loc[i_2-1, "discourse_end"] + 1
                end = df_essay.loc[i_2, 'discourse_start'] -1
                disc_type = "Nothing"
                gap_end = np.nan
                gap = np.nan
                df_essay.loc[insert_row] = [start, end, disc_type, gap, gap_end]
                insert_row += 1

    df_essay = df_essay.sort_values(by = "discourse_start").reset_index(drop=True)

    #add gap at end
    if df_essay.loc[(len(df_essay)-1),'gap_end_length'] > 0:
        start = df_essay.loc[(len(df_essay)-1), "discourse_end"] + 1
        end = start + df_essay.loc[(len(df_essay)-1), 'gap_end_length']
        disc_type = "Nothing"
        gap_end = np.nan
        gap = np.nan
        df_essay.loc[insert_row] = [start, end, disc_type, gap, gap_end]
        
    return(df_essay)
    
def print_colored_essay(essay):
    df_essay = add_gap_rows(essay)
    #code from https://www.kaggle.com/odins0n/feedback-prize-eda, but adjusted to df_essay
    essay_file = "/home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/input/feedback-prize-2021/train/" + essay + ".txt"

    ents = []
    for i, row in df_essay.iterrows():
        ents.append({
                        'start': int(row['discourse_start']), 
                         'end': int(row['discourse_end']), 
                         'label': row['discourse_type']
                    })

    with open(essay_file, 'r', encoding='utf-8') as file: data = file.read()

    doc2 = {
        "text": data,
        "ents": ents,
    }

    colors = {'Lead': '#EE11D0','Position': '#AB4DE1','Claim': '#1EDE71','Evidence': '#33FAFA','Counterclaim': '#4253C1','Concluding Statement': 'yellow','Rebuttal': 'red'}
    options = {"ents": df_essay.discourse_type.unique().tolist(), "colors": colors}

print_colored_essay(IREWR_plug_2)

In [ ]:

import cudf
try:
    pd_train = cudf.DataFrame(index=train.index).to_pandas()
except:
    pd_train = cudf.DataFrame().to_pandas()
                

In [ ]:
%%time
## Transfer_pre 21 ##
pd_train["""discourse_type"""] = train["""discourse_type"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 21 ##
pd_train["""discourse_text"""] = train["""discourse_text"""].to_numpy()

In [ ]:
%%time
### cell 21 ###

# lowercase the discourse text (runs on GPU via the cudf.pandas extension)
train['discourse_text'] = train['discourse_text'].str.lower()

# build the stopword list
stop_english = stopwords.words('english')
other_words_to_take_out = ['school', 'students', 'people', 'would', 'could', 'many']
stop_english.extend(other_words_to_take_out)

# split each discourse_text into words and explode so each word gets its own row
exploded = train.assign(Word=train['discourse_text'].str.split()).explode('Word')

# filter out stopwords (runs on GPU)
filtered = exploded[~exploded['Word'].isin(stop_english)]

# count frequencies of each word by discourse_type
freq = (
    filtered
    .groupby(['discourse_type', 'Word'])
    .size()
    .reset_index(name='Frequency')
)

# for each discourse_type, take the top 10 words by Frequency
top10 = (
    freq
    .sort_values(['discourse_type', 'Frequency'], ascending=[True, False])
    .groupby('discourse_type')
    .head(10)
)

# build counts_dict as a plain dict mapping each discourse_type to its small DataFrame
counts_dict = {
    dt: (
        group
        .drop('discourse_type', axis=1)
        .set_index('Word')
        .sort_values('Frequency', ascending=True)
    )
    for dt, group in top10.groupby('discourse_type')
}

# extract the keys and final df as in the original
keys = list(counts_dict.keys())
t_last = keys[-1]
df = train[train['discourse_type'] == t_last]

In [ ]:

import cudf
try:
    pd_df = cudf.DataFrame(index=df.index).to_pandas()
except:
    pd_df = cudf.DataFrame().to_pandas()
                

In [ ]:
%%time
## Transfer_post 21 ##
pd_df["""id"""] = df["""id"""].to_numpy()

In [ ]:
%%time
## Transfer_post 21 ##
pd_df["""discourse_id"""] = df["""discourse_id"""].to_numpy()

In [ ]:
%%time
## Transfer_post 21 ##
pd_df["""discourse_start"""] = df["""discourse_start"""].to_numpy()

In [ ]:
%%time
## Transfer_post 21 ##
pd_df["""discourse_end"""] = df["""discourse_end"""].to_numpy()

In [ ]:
%%time
## Transfer_post 21 ##
pd_df["""discourse_text"""] = df["""discourse_text"""].to_numpy()

In [ ]:
%%time
## Transfer_post 21 ##
pd_df["""discourse_type"""] = df["""discourse_type"""].to_numpy()

In [ ]:
%%time
## Transfer_post 21 ##
pd_df["""discourse_type_num"""] = df["""discourse_type_num"""].to_numpy()

In [ ]:
%%time
## Transfer_post 21 ##
pd_df["""predictionstring"""] = df["""predictionstring"""].to_numpy()

In [ ]:
%%time
## Transfer_post 21 ##
pd_df["""discourse_len"""] = df["""discourse_len"""].to_numpy()

In [ ]:
%%time
## Transfer_post 21 ##
pd_df["""pred_len"""] = df["""pred_len"""].to_numpy()

In [ ]:
%%time
## Transfer_post 21 ##
pd_df["""discourse_nr"""] = df["""discourse_nr"""].to_numpy()

In [ ]:
%%time
## Transfer_post 21 ##
pd_df["""essay_len"""] = df["""essay_len"""].to_numpy()

In [ ]:
%%time
## Transfer_post 21 ##
pd_df["""essay_words"""] = df["""essay_words"""].to_numpy()

In [ ]:
%%time
## Transfer_post 21 ##
pd_df["""gap_length"""] = df["""gap_length"""].to_numpy()

In [ ]:
%%time
## Transfer_post 21 ##
pd_df["""gap_end_length"""] = df["""gap_end_length"""].to_numpy()

In [ ]:

import cudf
try:
    pd_train = cudf.DataFrame(index=train.index).to_pandas()
except:
    pd_train = cudf.DataFrame().to_pandas()
                

In [ ]:
%%time
## Transfer_post 21 ##
pd_train["""discourse_text"""] = train["""discourse_text"""].to_numpy()

In [ ]:

import cudf
try:
    pd_train = cudf.DataFrame(index=train.index).to_pandas()
except:
    pd_train = cudf.DataFrame().to_pandas()
                

In [ ]:
%%time
## Transfer_pre 22 ##
pd_train["""discourse_type"""] = train["""discourse_type"""].to_numpy()

In [ ]:
%%time
### cell 22 ###

def get_n_grams(n_grams, top_n=10):
    dfs = []
    for dt in tqdm(train.discourse_type.unique()):
        # GPU‐accelerated boolean filtering
        df_dt = train[train.discourse_type == dt]
        texts = df_dt.discourse_text.tolist()

        # CountVectorizer still runs on CPU under the extension
        vec = CountVectorizer(
            lowercase=True,
            stop_words='english',
            ngram_range=(n_grams, n_grams)
        ).fit(texts)
        bag_of_words = vec.transform(texts)

        # aggregate counts
        sum_words = bag_of_words.sum(axis=0)
        words_freq = [(word, sum_words[0, idx])
                      for word, idx in vec.vocabulary_.items()]

        # build DataFrame exactly as original to preserve its index
        cvec_df = pd.DataFrame.from_records(
            words_freq,
            columns=['words', 'counts']
        ).sort_values(by='counts', ascending=False)
        cvec_df.insert(0, 'Discourse_type', dt)
        cvec_df = cvec_df.iloc[:top_n, :]

        dfs.append(cvec_df)

    # concatenate without resetting the index to match the original DataFrame's index
    return pd.concat(dfs)

bigrams = get_n_grams(n_grams=2, top_n=10)
bigrams.head()

In [ ]:

import cudf
try:
    pd_bigrams = cudf.DataFrame(index=bigrams.index).to_pandas()
except:
    pd_bigrams = cudf.DataFrame().to_pandas()
                

In [ ]:
%%time
## Transfer_post 22 ##
pd_bigrams["""Discourse_type"""] = bigrams["""Discourse_type"""].to_numpy()

In [ ]:
%%time
## Transfer_post 22 ##
pd_bigrams["""words"""] = bigrams["""words"""].to_numpy()

In [ ]:
%%time
## Transfer_post 22 ##
pd_bigrams["""counts"""] = bigrams["""counts"""].to_numpy()

In [ ]:

import cudf
try:
    pd_bigrams = cudf.DataFrame(index=bigrams.index).to_pandas()
except:
    pd_bigrams = cudf.DataFrame().to_pandas()
                

In [ ]:
%%time
## Transfer_pre 23 ##
pd_bigrams["""Discourse_type"""] = bigrams["""Discourse_type"""].to_numpy()

In [ ]:

import cudf
try:
    pd_data = cudf.DataFrame(index=data.index).to_pandas()
except:
    pd_data = cudf.DataFrame().to_pandas()
                

In [ ]:
%%time
## Transfer_pre 23 ##
pd_data["""discourse_type"""] = data["""discourse_type"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 23 ##
pd_data["""discourse_end"""] = data["""discourse_end"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 23 ##
pd_data["""discourse_start"""] = data["""discourse_start"""].to_numpy()

In [ ]:

import cudf
try:
    pd_df = cudf.DataFrame(index=df.index).to_pandas()
except:
    pd_df = cudf.DataFrame().to_pandas()
                

In [ ]:
%%time
## Transfer_pre 23 ##
pd_df["""id"""] = df["""id"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 23 ##
pd_df["""discourse_id"""] = df["""discourse_id"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 23 ##
pd_df["""discourse_start"""] = df["""discourse_start"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 23 ##
pd_df["""discourse_end"""] = df["""discourse_end"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 23 ##
pd_df["""discourse_text"""] = df["""discourse_text"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 23 ##
pd_df["""discourse_type"""] = df["""discourse_type"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 23 ##
pd_df["""discourse_type_num"""] = df["""discourse_type_num"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 23 ##
pd_df["""predictionstring"""] = df["""predictionstring"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 23 ##
pd_df["""discourse_len"""] = df["""discourse_len"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 23 ##
pd_df["""pred_len"""] = df["""pred_len"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 23 ##
pd_df["""discourse_nr"""] = df["""discourse_nr"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 23 ##
pd_df["""essay_len"""] = df["""essay_len"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 23 ##
pd_df["""essay_words"""] = df["""essay_words"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 23 ##
pd_df["""gap_length"""] = df["""gap_length"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 23 ##
pd_df["""gap_end_length"""] = df["""gap_end_length"""].to_numpy()

In [ ]:
%%time
### cell 23 ###

def plot_ngram(df, type="bigrams"):
    # Replace .query (CPU) with boolean indexing (GPU)
    data = df[df["Discourse_type"] == type][["words", "counts"]]
    data = data.set_index("words").sort_values(by="counts", ascending=True)
    # ... plotting logic goes here

plot_ngram(bigrams)

In [ ]:

import cudf
try:
    pd_data = cudf.DataFrame(index=data.index).to_pandas()
except:
    pd_data = cudf.DataFrame().to_pandas()
                

In [ ]:
%%time
## Transfer_pre 24 ##
pd_data["""discourse_type"""] = data["""discourse_type"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 24 ##
pd_data["""discourse_end"""] = data["""discourse_end"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 24 ##
pd_data["""discourse_start"""] = data["""discourse_start"""].to_numpy()

In [ ]:

import cudf
try:
    pd_df = cudf.DataFrame(index=df.index).to_pandas()
except:
    pd_df = cudf.DataFrame().to_pandas()
                

In [ ]:
%%time
## Transfer_pre 24 ##
pd_df["""id"""] = df["""id"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 24 ##
pd_df["""discourse_id"""] = df["""discourse_id"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 24 ##
pd_df["""discourse_start"""] = df["""discourse_start"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 24 ##
pd_df["""discourse_end"""] = df["""discourse_end"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 24 ##
pd_df["""discourse_text"""] = df["""discourse_text"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 24 ##
pd_df["""discourse_type"""] = df["""discourse_type"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 24 ##
pd_df["""discourse_type_num"""] = df["""discourse_type_num"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 24 ##
pd_df["""predictionstring"""] = df["""predictionstring"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 24 ##
pd_df["""discourse_len"""] = df["""discourse_len"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 24 ##
pd_df["""pred_len"""] = df["""pred_len"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 24 ##
pd_df["""discourse_nr"""] = df["""discourse_nr"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 24 ##
pd_df["""essay_len"""] = df["""essay_len"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 24 ##
pd_df["""essay_words"""] = df["""essay_words"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 24 ##
pd_df["""gap_length"""] = df["""gap_length"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 24 ##
pd_df["""gap_end_length"""] = df["""gap_end_length"""].to_numpy()

In [ ]:

import cudf
try:
    pd_train = cudf.DataFrame(index=train.index).to_pandas()
except:
    pd_train = cudf.DataFrame().to_pandas()
                

In [ ]:
%%time
## Transfer_pre 24 ##
pd_train["""discourse_type"""] = train["""discourse_type"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 24 ##
pd_train["""predictionstring"""] = train["""predictionstring"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 24 ##
pd_train["""id"""] = train["""id"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 24 ##
pd_train["""gap_end_length"""] = train["""gap_end_length"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 24 ##
pd_train["""discourse_id"""] = train["""discourse_id"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 24 ##
pd_train["""discourse_text"""] = train["""discourse_text"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 24 ##
pd_train["""gap_length"""] = train["""gap_length"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 24 ##
pd_train["""discourse_end"""] = train["""discourse_end"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 24 ##
pd_train["""discourse_type_num"""] = train["""discourse_type_num"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 24 ##
pd_train["""discourse_start"""] = train["""discourse_start"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 24 ##
pd_train["""discourse_len"""] = train["""discourse_len"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 24 ##
pd_train["""essay_words"""] = train["""essay_words"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 24 ##
pd_train["""pred_len"""] = train["""pred_len"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 24 ##
pd_train["""essay_len"""] = train["""essay_len"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 24 ##
pd_train["""discourse_nr"""] = train["""discourse_nr"""].to_numpy()

In [ ]:
%%time
### cell 24 ###

def get_n_grams(n_grams, top_n = 10):
    df_words = pd.DataFrame()
    for dt in tqdm(train['discourse_type'].unique()):
        df = train.query(f"discourse_type == '{dt}'")
        texts = df['discourse_text'].tolist()
        vec = CountVectorizer(lowercase = True, stop_words = 'english',\
                              ngram_range=(n_grams, n_grams)).fit(texts)
        bag_of_words = vec.transform(texts)
        sum_words = bag_of_words.sum(axis=0)
        words_freq = [(word, sum_words[0, idx]) for word, idx in vec.vocabulary_.items()]
        cvec_df = pd.DataFrame.from_records(words_freq,\
                                            columns= ['words', 'counts']).sort_values(by="counts", ascending=False)
        cvec_df.insert(0, "Discourse_type", dt)
        cvec_df = cvec_df.iloc[:top_n,:]
        df_words = df_words._append(cvec_df)
    return df_words
    
def plot_ngram(df, type = "bigrams"):
    for n, dt in enumerate(df.Discourse_type.unique()):
        data = df[df['Discourse_type'] == dt][['words', 'counts']].set_index("words").sort_values(by = "counts", ascending = True)
    
trigrams = get_n_grams(n_grams = 3, top_n=10)
plot_ngram(trigrams, type = "trigrams")

In [ ]:
%%time
### cell 25 ###

# https://www.kaggle.com/raghavendrakotala/fine-tunned-on-roberta-base-as-ner-problem-0-533
test_names, train_texts = [], []
for f in tqdm(list(os.listdir('/home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/input/feedback-prize-2021/train'))):
    test_names.append(f.replace('.txt', ''))
    train_texts.append(open('/home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/input/feedback-prize-2021/train/' + f, 'r', encoding='utf-8').read())
train_text_df = pd.DataFrame({'id': test_names, 'text': train_texts})
train_text_df.head()

In [ ]:

import cudf
try:
    pd_train_text_df = cudf.DataFrame(index=train_text_df.index).to_pandas()
except:
    pd_train_text_df = cudf.DataFrame().to_pandas()
                

In [ ]:
%%time
## Transfer_post 25 ##
pd_train_text_df["""id"""] = train_text_df["""id"""].to_numpy()

In [ ]:
%%time
## Transfer_post 25 ##
pd_train_text_df["""text"""] = train_text_df["""text"""].to_numpy()

In [ ]:

import cudf
try:
    pd_df = cudf.DataFrame(index=df.index).to_pandas()
except:
    pd_df = cudf.DataFrame().to_pandas()
                

In [ ]:
%%time
## Transfer_pre 26 ##
pd_df["""id"""] = df["""id"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 26 ##
pd_df["""discourse_id"""] = df["""discourse_id"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 26 ##
pd_df["""discourse_start"""] = df["""discourse_start"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 26 ##
pd_df["""discourse_end"""] = df["""discourse_end"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 26 ##
pd_df["""discourse_text"""] = df["""discourse_text"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 26 ##
pd_df["""discourse_type"""] = df["""discourse_type"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 26 ##
pd_df["""discourse_type_num"""] = df["""discourse_type_num"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 26 ##
pd_df["""predictionstring"""] = df["""predictionstring"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 26 ##
pd_df["""discourse_len"""] = df["""discourse_len"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 26 ##
pd_df["""pred_len"""] = df["""pred_len"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 26 ##
pd_df["""discourse_nr"""] = df["""discourse_nr"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 26 ##
pd_df["""essay_len"""] = df["""essay_len"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 26 ##
pd_df["""essay_words"""] = df["""essay_words"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 26 ##
pd_df["""gap_length"""] = df["""gap_length"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 26 ##
pd_df["""gap_end_length"""] = df["""gap_end_length"""].to_numpy()

In [ ]:

import cudf
try:
    pd_train = cudf.DataFrame(index=train.index).to_pandas()
except:
    pd_train = cudf.DataFrame().to_pandas()
                

In [ ]:
%%time
## Transfer_pre 26 ##
pd_train["""discourse_type"""] = train["""discourse_type"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 26 ##
pd_train["""id"""] = train["""id"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 26 ##
pd_train["""predictionstring"""] = train["""predictionstring"""].to_numpy()

In [ ]:

import cudf
try:
    pd_train_text_df = cudf.DataFrame(index=train_text_df.index).to_pandas()
except:
    pd_train_text_df = cudf.DataFrame().to_pandas()
                

In [ ]:
%%time
## Transfer_pre 26 ##
pd_train_text_df["""id"""] = train_text_df["""id"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 26 ##
pd_train_text_df["""text"""] = train_text_df["""text"""].to_numpy()

In [ ]:
%%time
### cell 26 ###

# 1. Tokenize each essay into one row per token, with token indices
tokenized = (
    train_text_df[['id', 'text']]
    .assign(tokens=train_text_df['text'].str.split(' '))
    .explode('tokens')
    .reset_index(drop=True)
)
tokenized['token_index'] = tokenized.groupby('id').cumcount()

# 2. Expand the discourse annotations into one row per token index with B-/I- labels
labels = (
    train[['id', 'discourse_type', 'predictionstring']]
    .reset_index()
    .rename(columns={'index': 'disc_row'})
    .assign(
        token_index=lambda df: df['predictionstring'].str.split(' ').apply(lambda lst: list(map(int, lst)))
    )
    .explode('token_index')
)

labels['pos_in_discourse'] = labels.groupby('disc_row').cumcount()
labels['label'] = (
    ('B-' + labels['discourse_type'])
    .where(labels['pos_in_discourse'] == 0,
           'I-' + labels['discourse_type'])
)
labels = labels[['id', 'token_index', 'label']]

# 3. Merge tokenized text with labels, defaulting missing labels to "O"
tokenized = (
    tokenized
    .merge(labels, on=['id', 'token_index'], how='left')
)
tokenized['label'] = tokenized['label'].fillna('O')

# 4. Aggregate back to one list of labels per essay
entities_df = (
    tokenized
    .groupby('id')['label']
    .agg(list)
    .reset_index()
    .rename(columns={'label': 'entities'})
)

# 5. Attach the entities column to the original DataFrame
train_text_df = train_text_df.merge(entities_df, on='id', how='left')

In [ ]:

import cudf
try:
    pd_train_text_df = cudf.DataFrame(index=train_text_df.index).to_pandas()
except:
    pd_train_text_df = cudf.DataFrame().to_pandas()
                

In [ ]:
%%time
## Transfer_pre 27 ##
pd_train_text_df["""id"""] = train_text_df["""id"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 27 ##
pd_train_text_df["""text"""] = train_text_df["""text"""].to_numpy()

In [ ]:
%%time
## Transfer_pre 27 ##
pd_train_text_df["""entities"""] = train_text_df["""entities"""].to_numpy()

In [ ]:
%%time
### cell 27 ###

train_text_df.head()